# 04 · Gold analytics — item performance

Combine sales and traffic into one row per item and compute a **conversion
rate** (`orders / pageviews`). This is the payoff of the medallion pipeline:
a clean, analytics-ready table built from two very different raw sources.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("04-gold-analytics").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| default catalog:", spark.conf.get("spark.sql.defaultCatalog"))

In [ ]:
from pyspark.sql.functions import coalesce, col, count, lit, sum, when

items = spark.table("demo.silver.items")
purchases = spark.table("demo.silver.purchases_enriched")
pageviews = spark.table("demo.silver.pageviews_by_items")

purchase_agg = purchases.groupBy("item_id").agg(
    sum("quantity").alias("items_sold"),
    count("id").alias("orders"),
    sum("total_price").alias("revenue"),
)
pageview_agg = (
    pageviews.filter(col("page") == "products")
    .groupBy("item_id").agg(count(lit(1)).alias("pageviews"))
)

gold = (
    items.select(col("id").alias("item_id"), col("name").alias("item_name"),
                 col("category").alias("item_category"))
    .join(purchase_agg, "item_id", "left")
    .join(pageview_agg, "item_id", "left")
    .select(
        "item_id", "item_name", "item_category",
        coalesce(col("items_sold"), lit(0)).cast("long").alias("items_sold"),
        coalesce(col("orders"), lit(0)).cast("long").alias("orders"),
        coalesce(col("revenue"), lit(0)).cast("decimal(20,2)").alias("revenue"),
        coalesce(col("pageviews"), lit(0)).cast("long").alias("pageviews"),
        when(coalesce(col("pageviews"), lit(0)) > 0, col("orders") / col("pageviews"))
            .otherwise(lit(0.0)).cast("double").alias("conversion_rate"),
    )
)
gold.write.format("iceberg").mode("overwrite").save("demo.gold.item_performance")
print("demo.gold.item_performance:", gold.count(), "rows")

## Top items by revenue

In [ ]:
%%sql
SELECT item_name, item_category, orders, revenue, pageviews,
       round(conversion_rate, 4) AS conversion_rate
FROM demo.gold.item_performance
ORDER BY revenue DESC
LIMIT 10

## Best converting items (with real traffic)

In [ ]:
%%sql
SELECT item_name, item_category, orders, pageviews,
       round(conversion_rate, 4) AS conversion_rate
FROM demo.gold.item_performance
WHERE pageviews > 0
ORDER BY conversion_rate DESC
LIMIT 10

## Revenue by category

In [ ]:
%%sql
SELECT item_category,
       sum(orders)   AS orders,
       sum(revenue)  AS revenue,
       sum(pageviews) AS pageviews
FROM demo.gold.item_performance
GROUP BY item_category
ORDER BY revenue DESC